# Task 2.2 — Compute Delay Propagation Risk

For each outbound flight, find the inbound flight using the same `aircraft_registration`:
- Join with `airline_sla` to get `min_turn_time` (narrow vs wide body)
- `available_turn_time = scheduled_departure − inbound_actual_arrival`
- If `available_turn_time < min_turn_time`: `propagated_delay = min_turn_time − available_turn_time`
- `risk_level`: **CRITICAL** if propagated_delay > 60 mins, **HIGH** if 30–60, **MEDIUM** if 15–30
- Write to `silver.delay_propagation_risk` — refreshed every 15 minutes


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"
SILVER_DIR = "/FileStore/delta_lake/silver"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_2_2_Delay_Propagation_Risk")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Read Required Tables

In [ ]:
# ── Read Silver flight_operations + Bronze reference tables ───────
flight_ops_df = spark.read.format("delta").load(f"{SILVER_DIR}/flight_operations")
aircraft_df = spark.read.format("delta").load(f"{BRONZE_DIR}/aircraft_master")
sla_df = spark.read.format("delta").load(f"{BRONZE_DIR}/airline_sla")

print(f"Flight operations: {flight_ops_df.count()}")
print(f"Aircraft master: {aircraft_df.count()}")
print(f"Airline SLA: {sla_df.count()}")


## Step 2 — Match Inbound → Outbound via Aircraft Registration

In [ ]:
# ── Find inbound-outbound pairs using same aircraft_registration ──
# Outbound flights: departures from the airport
outbound_df = (flight_ops_df
    .select(
        F.col("flight_id").alias("outbound_flight_id"),
        F.col("flight_number").alias("outbound_flight_number"),
        F.col("airline_code"),
        F.col("aircraft_registration"),
        F.col("scheduled_departure").alias("outbound_scheduled_departure"),
        F.col("operational_date")
    )
)

# Inbound flights: arrivals at the airport (use pushback_ts/actual times)
inbound_df = (flight_ops_df
    .select(
        F.col("flight_id").alias("inbound_flight_id"),
        F.col("flight_number").alias("inbound_flight_number"),
        F.col("aircraft_registration").alias("inbound_aircraft_reg"),
        F.col("scheduled_arrival").alias("inbound_scheduled_arrival"),
        F.col("pushback_ts").alias("inbound_actual_arrival"),
        F.col("delay_minutes").alias("inbound_delay_mins")
    )
)

# Join: match outbound to inbound on same aircraft_registration
# Inbound must arrive before outbound departs (same day or close)
paired_df = (outbound_df
    .join(inbound_df,
        (outbound_df.aircraft_registration == inbound_df.inbound_aircraft_reg) &
        (inbound_df.inbound_scheduled_arrival < outbound_df.outbound_scheduled_departure),
        "inner")
)

# Keep only the closest inbound flight per outbound (latest arrival before departure)
w = Window.partitionBy("outbound_flight_id").orderBy(F.col("inbound_scheduled_arrival").desc())
paired_df = paired_df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

print(f"Inbound-outbound pairs found: {paired_df.count()}")
paired_df.show(5, truncate=False)


## Step 3 — Compute Turn Time & Propagation Risk

In [ ]:
# ── Join aircraft body type and SLA min turn times ───────────────
paired_with_aircraft = paired_df.join(
    aircraft_df.select("aircraft_registration", "body_type"),
    on="aircraft_registration",
    how="left"
)

paired_with_sla = paired_with_aircraft.join(
    sla_df.select(
        "airline_code",
        "min_turn_time_narrow_body_mins",
        "min_turn_time_wide_body_mins"
    ),
    on="airline_code",
    how="left"
)

# Get the correct min_turn_time based on body type
delay_risk_df = paired_with_sla.withColumn(
    "min_turn_time_mins",
    F.when(F.col("body_type") == "WIDE", F.col("min_turn_time_wide_body_mins"))
     .otherwise(F.col("min_turn_time_narrow_body_mins"))
)

# Compute available turn time
delay_risk_df = delay_risk_df.withColumn(
    "available_turn_time_mins",
    F.round(
        (F.unix_timestamp("outbound_scheduled_departure") -
         F.unix_timestamp(F.coalesce("inbound_actual_arrival", "inbound_scheduled_arrival"))
        ) / 60, 1
    )
)

# Compute propagated delay
delay_risk_df = delay_risk_df.withColumn(
    "propagated_delay_mins",
    F.greatest(
        F.col("min_turn_time_mins") - F.col("available_turn_time_mins"),
        F.lit(0)
    )
)

# Assign risk_level
delay_risk_df = delay_risk_df.withColumn(
    "risk_level",
    F.when(F.col("propagated_delay_mins") > 60, "CRITICAL")
     .when(F.col("propagated_delay_mins") > 30, "HIGH")
     .when(F.col("propagated_delay_mins") > 15, "MEDIUM")
     .otherwise("LOW")
)

# Add alert timestamp
delay_risk_df = delay_risk_df.withColumn("alert_ts", F.current_timestamp())

# Filter to only at-risk flights (MEDIUM and above)
at_risk_df = delay_risk_df.filter(F.col("risk_level").isin("CRITICAL", "HIGH", "MEDIUM"))

print(f"Total paired flights analyzed: {delay_risk_df.count()}")
print(f"At-risk flights: {at_risk_df.count()}")
print("\nRisk Level Distribution:")
delay_risk_df.groupBy("risk_level").count().orderBy("count", ascending=False).show()


## Step 4 — Write to Silver Delta

In [ ]:
# ── Select final columns and write ────────────────────────────────
final_delay_risk_df = delay_risk_df.select(
    "outbound_flight_id", "outbound_flight_number",
    "inbound_flight_id", "inbound_flight_number",
    "airline_code", "aircraft_registration", "body_type",
    "outbound_scheduled_departure",
    "inbound_scheduled_arrival", "inbound_actual_arrival",
    "min_turn_time_mins", "available_turn_time_mins",
    "propagated_delay_mins", "risk_level",
    "operational_date", "alert_ts"
)

silver_delay_path = f"{SILVER_DIR}/delay_propagation_risk"
(final_delay_risk_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_delay_path))

print(f"Written to {silver_delay_path}")
result_df = spark.read.format("delta").load(silver_delay_path)
print(f"Silver delay_propagation_risk total: {result_df.count()}")
result_df.filter("risk_level != 'LOW'").orderBy("propagated_delay_mins", ascending=False).show(10, truncate=False)
